# CoreX V2 - Causal Graph Integration Training on Kaggle
**Feature Engineered RobotArm Anomaly Detection with Spatial Causal Graphs.**

Settings Required:
- GPU T4 x2

In [ ]:
%%bash
# === CELL 1: Copy Project ===
cd /kaggle/working
rm -rf project

SRC=$(find /kaggle/input -type d -name "Omni_Anomaly_Detection_coreX-main" | head -1)

if [ -n "$SRC" ]; then
    cp -r "$SRC" project
    echo "✅ Project copied from: $SRC"
    ls project/
else
    echo "❌ Could not find project folder. Listing input:"
    find /kaggle/input -maxdepth 3 -type d
fi

In [ ]:
%%bash
# === CELL 2: Fixed Environment Setup (From Yesterday's Working Run) ===
set -e

# 1. Install Miniconda if not present
if ! command -v /opt/conda/bin/conda &> /dev/null; then
    echo "=== Installing Miniconda ==="
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /opt/conda -f 2>/dev/null
fi
export PATH=/opt/conda/bin:$PATH

# 2. Accept TOS & Configure
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
conda config --set always_yes true

# 3. Create corex_env (Python 3.6)
echo "=== Creating corex_env ==="
conda env remove -n corex_env 2>/dev/null || true
conda create -n corex_env python=3.6 -y -q

# 4. Install TF 1.15 + CUDA + Data Stack
echo "=== Installing TF 1.15 + CUDA (This takes a few minutes) ==="
conda install -n corex_env -c defaults tensorflow-gpu=1.15 cudatoolkit=10.0 cudnn=7 numpy=1.16 scipy pandas scikit-learn matplotlib -y -q

# 5. Install specialized AI dependencies
PIP=/opt/conda/envs/corex_env/bin/pip
echo "=== Installing pip dependencies ==="
$PIP install -q \
    git+https://github.com/haowen-xu/tfsnippet.git@v0.2.0-alpha1 \
    git+https://github.com/thu-ml/zhusuan.git \
    seaborn \
    imageio

echo ""
echo "=== Verifying installs (Fixing Matplotlib Backend) ==="
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -c "
import tfsnippet; print('tfsnippet OK')
import zhusuan; print('zhusuan OK:', zhusuan.__version__)
import seaborn; print('seaborn OK')
"

echo "✅✅✅ ENVIRONMENT FULLY READY ✅✅✅"

In [ ]:
%%bash
# === CELL 3: Verify GPU ===
export PATH=/opt/conda/envs/corex_env/bin:$PATH
/opt/conda/envs/corex_env/bin/python -c "
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU Available:', tf.test.is_gpu_available())
print('GPU Name:', tf.test.gpu_device_name())
"

In [ ]:
%%bash
# === CELL 4: Install Missing Extras ===
/opt/conda/envs/corex_env/bin/pip install \
    tensorflow-probability==0.8.0 \
    tqdm==4.28.1 \
    fs==2.3.0 \
    click==7.0 \
    PyYAML==5.4.1 \
    xlrd==2.0.2 \
    openpyxl==3.1.3

In [ ]:
# === CELL 5: Patch main.py for Test Epochs ===
EPOCHS = 50
import os, re
os.chdir('/kaggle/working/project')

with open('main.py', 'r') as f:
    content = f.read()

content = re.sub(r'max_epoch\s*=\s*\d+', f'max_epoch = {EPOCHS}', content)
content = re.sub(r"'max_epoch':\s*\d+", f"'max_epoch': {EPOCHS}", content)
content = re.sub(r'batch_size\s*=\s*\d+', 'batch_size = 25', content)
content = re.sub(r"restore_dir\s*=\s*'[^']*'", 'restore_dir = None', content)

with open('main.py', 'w') as f:
    f.write(content)

print(f'✅ main.py patched: {EPOCHS} epochs, batch_size=25, fresh training')

# Optional: We don't necessarily NEED standardize_tf_imports.py running here since we are literally compiling TF 1.15,
# but the Causal Graph functions will naturally resolve over it seamlessly.

In [ ]:
%%bash
# === CELL 6: Preprocess Data (V2 Feature Engineering) ===
cd /kaggle/working/project
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u data_preprocess.py
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u feature_engineering.py

In [ ]:
%%bash
# === CELL 7: TRAIN WITH CAUSAL GRAPH ===
cd /kaggle/working/project

# Run standardize script to ensure any stray tf.compat.v1 imports are normalized for TF 1.15 backward compatibility
# (Even if not strictly needed in 1.15, it patches things cleanly)
if [ -f "../scripts/standardize_tf_imports.py" ]; then
    /opt/conda/envs/corex_env/bin/python -u ../scripts/standardize_tf_imports.py
elif [ -f "standardize_tf_imports.py" ]; then
    /opt/conda/envs/corex_env/bin/python -u standardize_tf_imports.py
fi

MPLBACKEND=Agg PYTHONPATH=. /opt/conda/envs/corex_env/bin/python -u main.py

In [ ]:
%%bash
# === CELL 8: PLOT RESULTS ===
cd /kaggle/working/project
if [ -f "plot_results.py" ]; then
    MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u plot_results.py
fi

In [ ]:
# === CELL 9: Export Results ===
import zipfile, os, glob

os.chdir('/kaggle/working/project')

def make_zip(name, patterns):
    with zipfile.ZipFile(f'/kaggle/working/{name}', 'w', zipfile.ZIP_DEFLATED) as z:
        for p in patterns:
            for f in glob.glob(p, recursive=True):
                z.write(f)
    print(f'✅ {name} created')

make_zip('coreX_v2_causal_results.zip', ['results/**/*'])
make_zip('coreX_v2_causal_model.zip', ['model_coreX_v2_optimized/**/*'])

print('\n🎉 Download your results from the Output tab!')